In [1]:
!pip install scikit-learn gradio pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 10.6 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 7.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 6.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 861.5/861.5 kB 273.4 kB/s  0:00:02eta 0:00:03
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [gradio]17/18 [gradio]face-hub]


In [2]:
!pip install scikit-learn gradio pandas

In [3]:
import pandas as pd

# A small, self-contained dataset so the project needs no external downloads
# or API keys. In a full-scale version this would be replaced with a larger
# dataset such as TMDB 5000 Movies.
movies = pd.DataFrame([
    {"title": "The Matrix", "genres": "Action Sci-Fi",
     "overview": "A computer hacker learns about the true nature of his reality and his role in the war against its controllers."},
    {"title": "Inception", "genres": "Action Sci-Fi Thriller",
     "overview": "A thief who steals corporate secrets through dream-sharing technology is given the inverse task of planting an idea into a target's subconscious."},
    {"title": "Interstellar", "genres": "Adventure Drama Sci-Fi",
     "overview": "A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival."},
    {"title": "The Dark Knight", "genres": "Action Crime Drama",
     "overview": "When the menace known as the Joker wreaks havoc on Gotham, Batman must accept one of the greatest psychological tests of his ability to fight injustice."},
    {"title": "Toy Story", "genres": "Animation Adventure Comedy",
     "overview": "A cowboy doll is threatened when a new spaceman action figure supplants him as top toy in a boy's room."},
    {"title": "Finding Nemo", "genres": "Animation Adventure Comedy",
     "overview": "After his son is captured, a timid clownfish sets out on a journey to bring him home."},
    {"title": "The Notebook", "genres": "Drama Romance",
     "overview": "A poor young man and a rich young woman fall in love, but societal differences and the passage of time complicate their romance."},
    {"title": "Titanic", "genres": "Drama Romance",
     "overview": "A young aristocrat falls in love with a poor artist aboard the luxurious, ill-fated ship."},
    {"title": "Get Out", "genres": "Horror Mystery Thriller",
     "overview": "A young African-American man visits his white girlfriend's family estate and uncovers a disturbing secret."},
    {"title": "A Quiet Place", "genres": "Horror Sci-Fi Thriller",
     "overview": "A family must live in silence while hiding from creatures that hunt anything they hear."},
    {"title": "The Avengers", "genres": "Action Adventure Sci-Fi",
     "overview": "Earth's mightiest heroes must come together to stop a mischievous god and his alien army from enslaving humanity."},
    {"title": "Guardians of the Galaxy", "genres": "Action Adventure Comedy Sci-Fi",
     "overview": "A group of intergalactic misfits must pull together to stop a fanatical warrior from taking control of the universe."},
])
movies["soup"] = movies["overview"] + " " + (movies["genres"] + " ") * 2  # genre weighted 2x
movies

,title,genres,overview,soup
0,The Matrix,Action Sci-Fi,A computer hacker learns about the true nature...,A computer hacker learns about the true nature...
1,Inception,Action Sci-Fi Thriller,A thief who steals corporate secrets through d...,A thief who steals corporate secrets through d...
2,Interstellar,Adventure Drama Sci-Fi,A team of explorers travel through a wormhole ...,A team of explorers travel through a wormhole ...
3,The Dark Knight,Action Crime Drama,When the menace known as the Joker wreaks havo...,When the menace known as the Joker wreaks havo...
4,Toy Story,Animation Adventure Comedy,A cowboy doll is threatened when a new spacema...,A cowboy doll is threatened when a new spacema...
5,Finding Nemo,Animation Adventure Comedy,"After his son is captured, a timid clownfish s...","After his son is captured, a timid clownfish s..."
6,The Notebook,Drama Romance,A poor young man and a rich young woman fall i...,A poor young man and a rich young woman fall i...
7,Titanic,Drama Romance,A young aristocrat falls in love with a poor a...,A young aristocrat falls in love with a poor a...
8,Get Out,Horror Mystery Thriller,A young African-American man visits his white ...,A young African-American man visits his white ...
9,A Quiet Place,Horror Sci-Fi Thriller,A family must live in silence while hiding fro...,A family must live in silence while hiding fro...


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF vectorization of the combined "soup" text (overview + weighted genres).
# This is the same core technique as the GeeksforGeeks tutorial this project
# is based on, applied here to the combined text rather than overview alone.
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["soup"])

# Cosine similarity between every pair of movies
similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Similarity matrix shape:", similarity_matrix.shape)

TF-IDF matrix shape: (12, 131)
Similarity matrix shape: (12, 12)


In [5]:
title_to_index = pd.Series(movies.index, index=movies["title"])

def recommend(title, top_n=5):
    if title not in title_to_index:
        return f"'{title}' was not found in the dataset. Try one of: {', '.join(movies['title'])}"

    idx = title_to_index[title]
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]  # exclude the movie itself

    input_genres = set(movies.loc[idx, "genres"].split())
    lines = [f"Because you liked {title}, here are {top_n} recommendations:\n"]
    for i, (movie_idx, score) in enumerate(scores, start=1):
        rec_title = movies.loc[movie_idx, "title"]
        rec_genres = set(movies.loc[movie_idx, "genres"].split())
        shared = input_genres & rec_genres
        reason = f"shares genres: {', '.join(shared)}" if shared else "similar plot themes"
        lines.append(f"{i}. {rec_title} (similarity: {score:.2f}) — {reason}")

    return "\n".join(lines)

# quick manual test
print(recommend("The Matrix"))

Because you liked The Matrix, here are 5 recommendations:

1. The Avengers (similarity: 0.27) — shares genres: Sci-Fi, Action
2. Guardians of the Galaxy (similarity: 0.26) — shares genres: Sci-Fi, Action
3. Inception (similarity: 0.24) — shares genres: Sci-Fi, Action
4. A Quiet Place (similarity: 0.19) — shares genres: Sci-Fi
5. Interstellar (similarity: 0.19) — shares genres: Sci-Fi


In [6]:
import gradio as gr

def gradio_recommend(title, top_n):
    return recommend(title, int(top_n))

demo = gr.Interface(
    fn=gradio_recommend,
    inputs=[
        gr.Dropdown(choices=list(movies["title"]), label="Pick a movie you like"),
        gr.Slider(minimum=1, maximum=8, value=5, step=1, label="Number of recommendations"),
    ],
    outputs=gr.Textbox(label="Recommendations", lines=10),
    title="Movie Recommendation System",
    description=(
        "A content-based recommender using TF-IDF and cosine similarity over "
        "movie overviews and genres, extending the approach described in "
        "GeeksforGeeks' movie recommender tutorial with genre weighting and "
        "an explanation of why each recommendation was chosen."
    ),
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
